# Rodent vs. No-Rodent Classifier — MobileNetV3-Small (Colab)

Fine-tunes an ImageNet-pretrained MobileNetV3-Small on your rodent / no-rodent dataset,
then exports a quantized TFLite model for edge deployment.

This is the MobileNetV3 counterpart of the earlier EfficientNet-Lite notebook. MobileNetV3
is a native `keras.applications` model (no TF-Hub involved), so it avoids both issues we hit
before: the Keras-3/TF-Hub incompatibility, and TF-Hub's EfficientNet-Lite modules blocking
fine-tuning entirely.

**Before running:** set `DATASET_ROOT` in the "Configure paths" cell below.

## 1. Setup

In [ ]:
!pip install -q scikit-learn ai-edge-litert


In [ ]:
import os
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ---- EDIT THIS ----
DATASET_ROOT = "/content/dataset/ei_curated"   # folder containing training/ and testing/
# --------------------

TRAIN_DIR = os.path.join(DATASET_ROOT, "training")
TEST_DIR  = os.path.join(DATASET_ROOT, "testing")

OUTPUT_DIR = "/content/drive/MyDrive/rodent_model_output_mobilenetv3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = 224          # MobileNetV3-Small native input size
BATCH_SIZE = 32
SEED = 42


## 3. Build a file list from the Edge Impulse export

Each of `training/` and `testing/` is a flat folder of images plus one `bounding_boxes.labels`
JSON file (the Edge Impulse bounding-box label format). An image counts as `rodent` if it has
at least one bounding box listed; otherwise (missing from the file, or an empty box list --
which includes your generated background images) it's `no_rodent`.

No grouping/leakage handling here, since the dataset is already cleaned and reduced.

In [ ]:
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

def parse_ei_labels(folder):
    """Reads bounding_boxes.labels from an Edge Impulse export folder and returns
    {filename: has_rodent_bool}."""
    labels_path = os.path.join(folder, "bounding_boxes.labels")
    with open(labels_path, "r") as f:
        data = json.load(f)

    boxes_by_file = data.get("boundingBoxes", data)  # tolerate slightly different EI export shapes
    has_rodent = {}
    for fname, boxes in boxes_by_file.items():
        has_rodent[fname] = bool(boxes) and len(boxes) > 0
    return has_rodent

def collect_files(folder):
    folder = Path(folder)
    has_rodent = parse_ei_labels(folder)

    rows = []
    for p in folder.iterdir():
        if p.suffix.lower() not in VALID_EXTS:
            continue
        is_rodent = has_rodent.get(p.name, False)
        rows.append({
            "filepath": str(p),
            "label": "rodent" if is_rodent else "no_rodent",
        })
    return pd.DataFrame(rows)

df_train_full = collect_files(TRAIN_DIR)
df_test = collect_files(TEST_DIR)

df_train_full["label_idx"] = (df_train_full["label"] == "rodent").astype(int)
df_test["label_idx"] = (df_test["label"] == "rodent").astype(int)

print("Training folder:")
print(df_train_full["label"].value_counts())
print("\nTesting folder:")
print(df_test["label"].value_counts())


## 4. Train / val split (testing/ is already your held-out test set)

A plain stratified random split of training/ into train and val -- no grouping, since the
dataset is already cleaned.

In [ ]:
df_train, df_val = train_test_split(
    df_train_full,
    test_size=0.15,
    stratify=df_train_full["label_idx"],
    random_state=SEED,
)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

print(f"Train: {len(df_train)} images")
print(f"Val:   {len(df_val)} images")
print(f"Test:  {len(df_test)} images")
print("\nClass balance (train):")
print(df_train['label'].value_counts(normalize=True))


## 5. Build tf.data pipelines

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def load_image(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    # Keep raw 0-255 float range -- MobileNetV3's keras.applications model has
    # include_preprocessing=True by default and normalizes internally.
    img = tf.cast(img, tf.float32)
    return img, label

def augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=25.0)   # scaled for 0-255 range
    img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label

def make_dataset(df_split, training=False):
    ds = tf.data.Dataset.from_tensor_slices((df_split["filepath"].values,
                                              df_split["label_idx"].values.astype(np.float32)))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.shuffle(1000, seed=SEED)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(df_train, training=True)
val_ds   = make_dataset(df_val,   training=False)
test_ds  = make_dataset(df_test,  training=False)


## 6. Build the model

`keras.applications.MobileNetV3Small` is a native Keras model (no TF-Hub), fully fine-tunable,
and well-suited for int8 TFLite Micro deployment. Swap `MobileNetV3Small` for `MobileNetV3Large`
below if you want more accuracy at the cost of a bigger model.

In [ ]:
def build_model(trainable_backbone=False):
    base_model = tf.keras.applications.MobileNetV3Small(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg",
        include_preprocessing=True,   # normalizes raw [0,255] input internally
    )
    base_model.trainable = trainable_backbone

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        base_model,
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    return model

model = build_model(trainable_backbone=False)
model.summary()


## 7. Phase 1 — train the classification head (backbone frozen)

In [ ]:
n_rodent = (df_train["label_idx"] == 1).sum()
n_no_rodent = (df_train["label_idx"] == 0).sum()
total = n_rodent + n_no_rodent
class_weight = {
    0: total / (2 * n_no_rodent),
    1: total / (2 * n_rodent),
}
print("Class weights:", class_weight)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase1_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight,
    callbacks=callbacks,
)


## 8. (Optional) Phase 2 — fine-tune the backbone

Only run this if phase-1 accuracy/precision/recall (section 9, evaluated on `model`) isn't good
enough. Unfreezes the backbone and continues training with a much lower learning rate. If you
skip this cell, just use `model` (not `model_ft`) in sections 9-10 below.

In [ ]:
model_ft = build_model(trainable_backbone=True)
model_ft.set_weights(model.get_weights())  # carry over phase-1 weights

model_ft.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
             tf.keras.metrics.Recall(name="recall")]
)

callbacks_ft = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(os.path.join(OUTPUT_DIR, "phase2_best.keras"),
                                        monitor="val_loss", save_best_only=True),
]

history2 = model_ft.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight,
    callbacks=callbacks_ft,
)


## 9. Evaluate on the held-out test set

Set `FINAL_MODEL` below to whichever model you actually want to evaluate/export: `model`
(phase 1 only) or `model_ft` (if you ran phase 2).

In [ ]:
FINAL_MODEL = model   # change to model_ft if you ran phase 2

test_loss, test_acc, test_prec, test_rec = FINAL_MODEL.evaluate(test_ds)
print(f"Test accuracy:  {test_acc:.4f}")
print(f"Test precision: {test_prec:.4f}")
print(f"Test recall:    {test_rec:.4f}")

y_true = df_test["label_idx"].values
y_pred_prob = FINAL_MODEL.predict(test_ds).ravel()
y_pred = (y_pred_prob > 0.5).astype(int)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["no_rodent", "rodent"]))

print("Confusion matrix (rows=true, cols=pred), order [no_rodent, rodent]:")
print(confusion_matrix(y_true, y_pred))


## 10. Convert to TFLite and quantize

`dynamic_range` quantization is a safe default. For a microcontroller target, use the
`full_integer` cell (Option B), which needs a representative dataset sample.

In [ ]:
saved_model_dir = os.path.join(OUTPUT_DIR, "saved_model")
FINAL_MODEL.export(saved_model_dir)

# --- Option A: dynamic-range quantization (good default) ---
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic = converter.convert()

with open(os.path.join(OUTPUT_DIR, "model_dynamic_range.tflite"), "wb") as f:
    f.write(tflite_dynamic)

print("Saved model_dynamic_range.tflite —", len(tflite_dynamic) / 1024, "KB")


In [ ]:
# --- Option B: full integer quantization (for microcontrollers / strict edge targets) ---
def representative_dataset():
    for images, _ in train_ds.take(50):
        for img in images:
            yield [tf.expand_dims(img, 0)]

converter_int8 = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type = tf.uint8
converter_int8.inference_output_type = tf.uint8

tflite_int8 = converter_int8.convert()

with open(os.path.join(OUTPUT_DIR, "model_int8.tflite"), "wb") as f:
    f.write(tflite_int8)

print("Saved model_int8.tflite —", len(tflite_int8) / 1024, "KB")


## 11. Verify the quantized model still performs well

Uses the newer `ai_edge_litert` interpreter (the `tf.lite.Interpreter` API is being phased out
of TensorFlow starting TF 2.20).

In [ ]:
from ai_edge_litert.interpreter import Interpreter

def evaluate_tflite(tflite_path, df_split, is_int8=False):
    interpreter = Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    for _, row in df_split.iterrows():
        img = tf.io.read_file(row["filepath"])
        img = tf.image.decode_image(img, channels=3, expand_animations=False)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = tf.cast(img, tf.float32)   # raw 0-255, matches the training pipeline
        if is_int8:
            img = tf.cast(img, tf.uint8)
        img = tf.expand_dims(img, 0)

        interpreter.set_tensor(input_details["index"], img)
        interpreter.invoke()
        pred = interpreter.get_tensor(output_details["index"])[0][0]
        if is_int8:
            pred = pred / 255.0
        pred_label = 1 if pred > 0.5 else 0
        correct += int(pred_label == row["label_idx"])

    return correct / len(df_split)

acc_dynamic = evaluate_tflite(os.path.join(OUTPUT_DIR, "model_dynamic_range.tflite"), df_test, is_int8=False)
print(f"Dynamic-range TFLite test accuracy: {acc_dynamic:.4f}")

acc_int8 = evaluate_tflite(os.path.join(OUTPUT_DIR, "model_int8.tflite"), df_test, is_int8=True)
print(f"Int8 TFLite test accuracy: {acc_int8:.4f}")


## 12. Save labels file (needed alongside the .tflite model for inference)

In [ ]:
labels = ["no_rodent", "rodent"]
with open(os.path.join(OUTPUT_DIR, "labels.txt"), "w") as f:
    f.write("\n".join(labels))

print("All outputs saved to:", OUTPUT_DIR)
print(os.listdir(OUTPUT_DIR))


## 13. Package the int8 model as an Arduino library

Converts `model_int8.tflite` into a C byte array and assembles an Arduino library folder
(`library.properties`, `src/`, an example sketch), then zips it — ready to install via
Arduino IDE's *Sketch → Include Library → Add .ZIP Library*.

**One real dependency you still need to install yourself:** the
[TensorFlow Lite for Microcontrollers Arduino library](https://github.com/tensorflow/tflite-micro-arduino-examples)
(search "Arduino_TensorFlowLite" or "Chirale_TensorFlowLite" in the Arduino Library Manager).
Your generated library depends on it at compile time.

In [ ]:
LIBRARY_NAME = "RodentDetectorV3"   # change if you want a different library/class name

lib_root = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
src_dir = os.path.join(lib_root, "src")
example_dir = os.path.join(lib_root, "examples", f"{LIBRARY_NAME}_Inference")

os.makedirs(src_dir, exist_ok=True)
os.makedirs(example_dir, exist_ok=True)


### 13a. Convert the .tflite file into a C byte array

In [ ]:
def tflite_to_c_array(tflite_path, var_name="g_model"):
    with open(tflite_path, "rb") as f:
        data = f.read()

    lines = []
    for i in range(0, len(data), 12):
        chunk = data[i:i + 12]
        lines.append(", ".join(f"0x{b:02x}" for b in chunk))
    body = ",\n  ".join(lines)

    header = f"""// Auto-generated from {os.path.basename(tflite_path)} -- do not edit by hand.
#ifndef {var_name.upper()}_DATA_H
#define {var_name.upper()}_DATA_H

alignas(8) const unsigned char {var_name}[] = {{
  {body}
}};
const unsigned int {var_name}_len = {len(data)};

#endif  // {var_name.upper()}_DATA_H
"""
    return header, len(data)

model_header, model_size = tflite_to_c_array(
    os.path.join(OUTPUT_DIR, "model_int8.tflite"), var_name="g_model"
)

with open(os.path.join(src_dir, "model_data.h"), "w") as f:
    f.write(model_header)

print(f"Embedded model: {model_size} bytes ({model_size/1024:.1f} KB) -> src/model_data.h")


### 13b. Write library.properties

In [ ]:
library_properties = f"""name={LIBRARY_NAME}
version=1.0.0
author=Your Name
maintainer=Your Name <you@example.com>
sentence=On-device rodent / no-rodent classification using a quantized MobileNetV3-Small model.
paragraph=Runs TensorFlow Lite for Microcontrollers inference on a quantized int8 MobileNetV3-Small model to classify camera frames as rodent or no_rodent.
category=Sensors
url=
architectures=*
depends=Arduino_TensorFlowLite
"""

with open(os.path.join(lib_root, "library.properties"), "w") as f:
    f.write(library_properties)


### 13c. Write the library header/source

A thin wrapper class around the TFLite Micro interpreter. It expects a `{IMG_SIZE}x{IMG_SIZE}x3`
uint8 image buffer as input and returns a float probability (rodent likelihood) as output.

In [ ]:
header_code = f"""#ifndef {LIBRARY_NAME.upper()}_H
#define {LIBRARY_NAME.upper()}_H

#include <TensorFlowLite.h>

class {LIBRARY_NAME} {{
  public:
    {LIBRARY_NAME}();
    bool begin();
    // input: pointer to {IMG_SIZE}*{IMG_SIZE}*3 uint8 RGB bytes, already resized to model input size
    // returns: probability that the image contains a rodent (0.0 - 1.0)
    float predict(const uint8_t* image_data);

  private:
    bool _initialized = false;
}};

#endif  // {LIBRARY_NAME.upper()}_H
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.h"), "w") as f:
    f.write(header_code)


In [ ]:
cpp_code = f"""#include "{LIBRARY_NAME}.h"
#include "model_data.h"

#include <tensorflow/lite/micro/micro_interpreter.h>
#include <tensorflow/lite/micro/micro_mutable_op_resolver.h>
#include <tensorflow/lite/micro/micro_log.h>
#include <tensorflow/lite/schema/schema_generated.h>

namespace {{
  constexpr int kTensorArenaSize = 200 * 1024;  // tune down/up depending on your board's RAM
  alignas(16) uint8_t tensor_arena[kTensorArenaSize];

  const tflite::Model* model = nullptr;
  tflite::MicroInterpreter* interpreter = nullptr;
  TfLiteTensor* input = nullptr;
  TfLiteTensor* output = nullptr;
}}

{LIBRARY_NAME}::{LIBRARY_NAME}() {{}}

bool {LIBRARY_NAME}::begin() {{
  model = tflite::GetModel(g_model);
  if (model->version() != TFLITE_SCHEMA_VERSION) {{
    MicroPrintf("Model schema version mismatch!");
    return false;
  }}

  // Add only the ops your model actually needs (keeps flash usage down).
  // If you get an "unresolved op" error at runtime, add the missing op here.
  static tflite::MicroMutableOpResolver<10> resolver;
  resolver.AddConv2D();
  resolver.AddDepthwiseConv2D();
  resolver.AddFullyConnected();
  resolver.AddLogistic();
  resolver.AddAveragePool2D();
  resolver.AddReshape();
  resolver.AddAdd();
  resolver.AddMul();
  resolver.AddPad();
  resolver.AddMean();

  static tflite::MicroInterpreter static_interpreter(
      model, resolver, tensor_arena, kTensorArenaSize);
  interpreter = &static_interpreter;

  if (interpreter->AllocateTensors() != kTfLiteOk) {{
    MicroPrintf("AllocateTensors() failed");
    return false;
  }}

  input = interpreter->input(0);
  output = interpreter->output(0);
  _initialized = true;
  return true;
}}

float {LIBRARY_NAME}::predict(const uint8_t* image_data) {{
  if (!_initialized) return -1.0f;

  memcpy(input->data.uint8, image_data, input->bytes);

  if (interpreter->Invoke() != kTfLiteOk) {{
    MicroPrintf("Invoke() failed");
    return -1.0f;
  }}

  uint8_t raw = output->data.uint8[0];
  float scale = output->params.scale;
  int zero_point = output->params.zero_point;
  return (raw - zero_point) * scale;
}}
"""

with open(os.path.join(src_dir, f"{LIBRARY_NAME}.cpp"), "w") as f:
    f.write(cpp_code)


### 13d. Write an example sketch

In [ ]:
example_sketch = f"""#include <{LIBRARY_NAME}.h>

{LIBRARY_NAME} detector;

uint8_t image_buffer[{IMG_SIZE} * {IMG_SIZE} * 3];

void setup() {{
  Serial.begin(115200);
  while (!Serial) {{}}

  if (!detector.begin()) {{
    Serial.println("Failed to initialize model.");
    while (1) {{}}
  }}
  Serial.println("Model ready.");
}}

void loop() {{
  // TODO: capture a frame into image_buffer here, resized to {IMG_SIZE}x{IMG_SIZE}
  float rodent_prob = detector.predict(image_buffer);
  Serial.print("Rodent probability: ");
  Serial.println(rodent_prob);

  delay(1000);
}}
"""

with open(os.path.join(example_dir, f"{LIBRARY_NAME}_Inference.ino"), "w") as f:
    f.write(example_sketch)


### 13e. Zip the library

In [ ]:
zip_path_base = os.path.join(OUTPUT_DIR, LIBRARY_NAME)
zip_path = shutil.make_archive(zip_path_base, "zip", root_dir=OUTPUT_DIR, base_dir=LIBRARY_NAME)

print("Arduino library zipped at:", zip_path)
print("\nInstall via Arduino IDE: Sketch -> Include Library -> Add .ZIP Library... -> select this file.")
print("Remember to also install the 'Arduino_TensorFlowLite' (or 'Chirale_TensorFlowLite') library")
print("from the Library Manager first -- this generated library depends on it.")
